# Prometheus Grafana Monitor — Tutorial & Guide

This notebook is an interactive companion to the Prometheus Grafana Monitor weave. It teaches you how to:

1. **Understand** the monitoring stack's architecture and what each component does
2. **Query** Prometheus metrics programmatically using the HTTP API
3. **Visualize** CPU, memory, disk, and network data with matplotlib
4. **Build** custom analyses beyond what the Grafana dashboard provides
5. **Extend** the monitoring stack for your own experiments

**Prerequisites**: The weave must be running (or have run recently) so that `results/monitoring-status.json` exists with node IPs. If the weave is currently active, you can query live metrics.

---
## 1. How the Monitoring Stack Works

### Components

| Component | Role | Runs On | Port |
|-----------|------|---------|------|
| **Prometheus** | Time-series database — scrapes and stores metrics | monitor | 9090 |
| **Grafana** | Dashboard UI — visualizes Prometheus data | monitor | 3000 |
| **node_exporter** | Agent — exposes OS/hardware metrics | all nodes | 9100 |

### Data Flow

```
node_exporter ──► /metrics endpoint ──► Prometheus (scrapes every 15s) ──► Grafana (queries on demand)
                  (port 9100)            (stores time-series)              (renders dashboards)
```

**Prometheus pull model**: Unlike push-based monitoring (e.g., StatsD), Prometheus *pulls* metrics from targets. Each node runs node_exporter, which collects OS metrics and serves them at `http://<ip>:9100/metrics`. Prometheus periodically scrapes this endpoint and stores the results.

**PromQL**: Prometheus has its own query language (PromQL) for selecting and aggregating time-series data. You'll use it both in Grafana dashboards and in the API queries below.

### Metric Types

node_exporter provides several types of metrics:

| Type | Example | How to Use |
|------|---------|------------|
| **Counter** | `node_cpu_seconds_total` | Always increasing. Use `rate()` to get per-second change |
| **Gauge** | `node_memory_MemAvailable_bytes` | Current value. Use directly or compute ratios |
| **Histogram** | (not used by node_exporter) | Distribution of values in buckets |

---
## 2. Loading the Monitoring Status

The weave saves connection details to `results/monitoring-status.json` after deployment. Let's load it.

In [ ]:
import json
import os
import urllib.request
import urllib.parse
from datetime import datetime, timezone

# Load monitoring status
STATUS_DIR = os.path.join(os.path.dirname(os.getcwd()), 'results')
if not os.path.exists(STATUS_DIR):
    STATUS_DIR = '../results'

status_path = os.path.join(STATUS_DIR, 'monitoring-status.json')
if os.path.exists(status_path):
    with open(status_path) as f:
        status = json.load(f)
    
    SLICE_NAME = status.get('slice_name', '?')
    NODE_IPS = status.get('node_ips', {})
    MONITOR_IP = status.get('monitor_ip', 'localhost')
    PROM_URL = status.get('prometheus_url', f'http://{MONITOR_IP}:9090')
    GRAFANA_URL = status.get('grafana_url', f'http://{MONITOR_IP}:3000')
    
    print(f'Slice: {SLICE_NAME}')
    print(f'\nNode IPs (FABNetv4):')
    for name, ip in NODE_IPS.items():
        role = 'monitor' if name == 'monitor' else 'worker'
        print(f'  {name} ({role}): {ip}')
    print(f'\nPrometheus: {PROM_URL}')
    print(f'Grafana:    {GRAFANA_URL}')
    
    tunnel = status.get('tunnel')
    if tunnel:
        print(f'Tunnel:     port {tunnel.get("local_port", "?")}')
else:
    print('No monitoring-status.json found.')
    print('Run the weave first, or set MONITOR_IP manually:')
    print('  MONITOR_IP = "10.x.x.x"')
    print('  PROM_URL = f"http://{MONITOR_IP}:9090"')
    MONITOR_IP = None
    PROM_URL = None
    NODE_IPS = {}

---
## 3. Querying Prometheus

Prometheus provides an HTTP API for querying metrics. The two main endpoints are:

| Endpoint | Purpose | Returns |
|----------|---------|--------|
| `/api/v1/query` | Instant query (current value) | Single value per time-series |
| `/api/v1/query_range` | Range query (over time) | Time-series with multiple data points |

Let's create helper functions for both.

In [ ]:
def prom_query(query, prom_url=None):
    """Execute an instant PromQL query. Returns the result list."""
    url = prom_url or PROM_URL
    if not url:
        print('PROM_URL not set. Run the weave first.')
        return []
    
    params = urllib.parse.urlencode({'query': query})
    req_url = f'{url}/api/v1/query?{params}'
    try:
        with urllib.request.urlopen(req_url, timeout=10) as resp:
            data = json.loads(resp.read())
        if data.get('status') == 'success':
            return data['data']['result']
        else:
            print(f'Query error: {data.get("error", "unknown")}')
            return []
    except Exception as e:
        print(f'Could not reach Prometheus at {url}: {e}')
        return []


def prom_query_range(query, start=None, end=None, step='15s', prom_url=None):
    """Execute a range PromQL query. Returns the result list with time-series values."""
    url = prom_url or PROM_URL
    if not url:
        print('PROM_URL not set. Run the weave first.')
        return []
    
    now = datetime.now(timezone.utc)
    if end is None:
        end = now.timestamp()
    if start is None:
        start = end - 3600  # Default: last 1 hour
    
    params = urllib.parse.urlencode({
        'query': query,
        'start': start,
        'end': end,
        'step': step,
    })
    req_url = f'{url}/api/v1/query_range?{params}'
    try:
        with urllib.request.urlopen(req_url, timeout=10) as resp:
            data = json.loads(resp.read())
        if data.get('status') == 'success':
            return data['data']['result']
        else:
            print(f'Query error: {data.get("error", "unknown")}')
            return []
    except Exception as e:
        print(f'Could not reach Prometheus at {url}: {e}')
        return []


print('Helper functions defined: prom_query(), prom_query_range()')

### 3a. Check Prometheus Targets

Let's verify that Prometheus is scraping all nodes successfully.

In [ ]:
if PROM_URL:
    try:
        with urllib.request.urlopen(f'{PROM_URL}/api/v1/targets', timeout=10) as resp:
            targets_data = json.loads(resp.read())
        
        active = targets_data.get('data', {}).get('activeTargets', [])
        print(f'Active scrape targets: {len(active)}')
        print(f'{"Job":<12} {"Instance":<25} {"Health":<8} {"Last Scrape"}')
        print('-' * 65)
        for t in active:
            labels = t.get('labels', {})
            health = t.get('health', 'unknown')
            last = t.get('lastScrape', '?')
            if isinstance(last, str) and len(last) > 19:
                last = last[:19]  # Trim to readable length
            print(f'{labels.get("job", "?"):<12} {labels.get("instance", "?"):<25} {health:<8} {last}')
    except Exception as e:
        print(f'Could not reach Prometheus: {e}')
else:
    print('PROM_URL not set. Run the weave first.')

### 3b. Instant Queries — Current Values

Let's query some current metrics.

In [ ]:
if PROM_URL:
    # --- CPU utilization (current, per node) ---
    print('=== Current CPU Utilization ===')
    results = prom_query('100 - (avg by(instance) (rate(node_cpu_seconds_total{mode="idle"}[5m])) * 100)')
    for r in results:
        instance = r['metric'].get('instance', '?')
        value = float(r['value'][1])
        print(f'  {instance:<25} {value:>6.1f}%')
    print()
    
    # --- Memory utilization (current, per node) ---
    print('=== Current Memory Utilization ===')
    results = prom_query(
        '(1 - (node_memory_MemAvailable_bytes / node_memory_MemTotal_bytes)) * 100'
    )
    for r in results:
        instance = r['metric'].get('instance', '?')
        value = float(r['value'][1])
        print(f'  {instance:<25} {value:>6.1f}%')
    print()
    
    # --- System load ---
    print('=== System Load (1m average) ===')
    results = prom_query('node_load1')
    for r in results:
        instance = r['metric'].get('instance', '?')
        value = float(r['value'][1])
        print(f'  {instance:<25} {value:>6.2f}')
    print()
    
    # --- Disk usage (root filesystem) ---
    print('=== Disk Usage (/) ===')
    results = prom_query(
        '(1 - (node_filesystem_avail_bytes{mountpoint="/",fstype!="rootfs"} '
        '/ node_filesystem_size_bytes{mountpoint="/",fstype!="rootfs"})) * 100'
    )
    for r in results:
        instance = r['metric'].get('instance', '?')
        value = float(r['value'][1])
        print(f'  {instance:<25} {value:>6.1f}%')
    print()
    
    # --- Uptime ---
    print('=== Uptime ===')
    results = prom_query('time() - node_boot_time_seconds')
    for r in results:
        instance = r['metric'].get('instance', '?')
        seconds = float(r['value'][1])
        hours = int(seconds // 3600)
        minutes = int((seconds % 3600) // 60)
        print(f'  {instance:<25} {hours}h {minutes}m')
else:
    print('PROM_URL not set.')

---
## 4. Visualizing Metrics with matplotlib

While Grafana handles real-time dashboards, matplotlib gives you full control for custom analysis, publication-quality figures, and offline work.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib
import numpy as np

matplotlib.rcParams['figure.figsize'] = (12, 5)
matplotlib.rcParams['figure.dpi'] = 100

print('matplotlib ready')

### 4a. CPU Utilization Over Time

In [ ]:
if PROM_URL:
    # Query: CPU utilization over the last hour, sampled every 30s
    results = prom_query_range(
        '100 - (avg by(instance) (rate(node_cpu_seconds_total{mode="idle"}[5m])) * 100)',
        step='30s'
    )
    
    if results:
        fig, ax = plt.subplots(figsize=(12, 5))
        for series in results:
            instance = series['metric'].get('instance', '?')
            # Extract short name (remove port)
            label = instance.split(':')[0]
            
            timestamps = [datetime.fromtimestamp(float(v[0])) for v in series['values']]
            values = [float(v[1]) for v in series['values']]
            
            ax.plot(timestamps, values, label=label, linewidth=1.5)
        
        ax.set_xlabel('Time')
        ax.set_ylabel('CPU Utilization (%)')
        ax.set_title('CPU Utilization by Node (Last Hour)')
        ax.set_ylim(0, 100)
        ax.legend()
        ax.grid(alpha=0.3)
        fig.autofmt_xdate()
        plt.tight_layout()
        plt.show()
    else:
        print('No CPU data returned. Is Prometheus running?')
else:
    print('PROM_URL not set.')

### 4b. Memory Utilization Over Time

In [ ]:
if PROM_URL:
    results = prom_query_range(
        '(1 - (node_memory_MemAvailable_bytes / node_memory_MemTotal_bytes)) * 100',
        step='30s'
    )
    
    if results:
        fig, ax = plt.subplots(figsize=(12, 5))
        for series in results:
            instance = series['metric'].get('instance', '?')
            label = instance.split(':')[0]
            
            timestamps = [datetime.fromtimestamp(float(v[0])) for v in series['values']]
            values = [float(v[1]) for v in series['values']]
            
            ax.plot(timestamps, values, label=label, linewidth=1.5)
        
        ax.set_xlabel('Time')
        ax.set_ylabel('Memory Utilization (%)')
        ax.set_title('Memory Utilization by Node (Last Hour)')
        ax.set_ylim(0, 100)
        ax.legend()
        ax.grid(alpha=0.3)
        fig.autofmt_xdate()
        plt.tight_layout()
        plt.show()
    else:
        print('No memory data returned.')
else:
    print('PROM_URL not set.')

### 4c. Network Traffic Over Time

In [ ]:
if PROM_URL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Receive traffic
    results = prom_query_range(
        'sum by(instance) (rate(node_network_receive_bytes_total{device!~"lo|veth.*|docker.*|br-.*"}[5m]))',
        step='30s'
    )
    for series in results:
        instance = series['metric'].get('instance', '?')
        label = instance.split(':')[0]
        timestamps = [datetime.fromtimestamp(float(v[0])) for v in series['values']]
        values = [float(v[1]) / 1e6 for v in series['values']]  # Convert to MB/s
        ax1.plot(timestamps, values, label=label, linewidth=1.5)
    
    ax1.set_xlabel('Time')
    ax1.set_ylabel('Receive (MB/s)')
    ax1.set_title('Network Receive by Node')
    ax1.legend(fontsize=9)
    ax1.grid(alpha=0.3)
    
    # Transmit traffic
    results = prom_query_range(
        'sum by(instance) (rate(node_network_transmit_bytes_total{device!~"lo|veth.*|docker.*|br-.*"}[5m]))',
        step='30s'
    )
    for series in results:
        instance = series['metric'].get('instance', '?')
        label = instance.split(':')[0]
        timestamps = [datetime.fromtimestamp(float(v[0])) for v in series['values']]
        values = [float(v[1]) / 1e6 for v in series['values']]
        ax2.plot(timestamps, values, label=label, linewidth=1.5)
    
    ax2.set_xlabel('Time')
    ax2.set_ylabel('Transmit (MB/s)')
    ax2.set_title('Network Transmit by Node')
    ax2.legend(fontsize=9)
    ax2.grid(alpha=0.3)
    
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()
else:
    print('PROM_URL not set.')

### 4d. System Load Comparison

In [ ]:
if PROM_URL:
    fig, ax = plt.subplots(figsize=(12, 5))
    
    colors = {'1': '#4C72B0', '5': '#DD8452', '15': '#55A868'}
    
    for load_avg, color in [('1', colors['1']), ('5', colors['5']), ('15', colors['15'])]:
        results = prom_query_range(f'node_load{load_avg}', step='30s')
        for i, series in enumerate(results):
            instance = series['metric'].get('instance', '?').split(':')[0]
            timestamps = [datetime.fromtimestamp(float(v[0])) for v in series['values']]
            values = [float(v[1]) for v in series['values']]
            
            # Only show the load label once in legend
            label = f'load{load_avg} ({instance})'
            ax.plot(timestamps, values, label=label, color=color,
                    linestyle=['-', '--', ':'][i % 3], linewidth=1.5, alpha=0.8)
    
    ax.set_xlabel('Time')
    ax.set_ylabel('Load Average')
    ax.set_title('System Load Average by Node (Last Hour)')
    ax.legend(fontsize=8, ncol=3)
    ax.grid(alpha=0.3)
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()
else:
    print('PROM_URL not set.')

### 4e. Disk I/O Over Time

In [ ]:
if PROM_URL:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # Disk read
    results = prom_query_range(
        'sum by(instance) (rate(node_disk_read_bytes_total{device!~"dm-.*"}[5m]))',
        step='30s'
    )
    for series in results:
        instance = series['metric'].get('instance', '?').split(':')[0]
        timestamps = [datetime.fromtimestamp(float(v[0])) for v in series['values']]
        values = [float(v[1]) / 1e6 for v in series['values']]  # MB/s
        ax1.plot(timestamps, values, label=instance, linewidth=1.5)
    
    ax1.set_xlabel('Time')
    ax1.set_ylabel('Read (MB/s)')
    ax1.set_title('Disk Read I/O by Node')
    ax1.legend(fontsize=9)
    ax1.grid(alpha=0.3)
    
    # Disk write
    results = prom_query_range(
        'sum by(instance) (rate(node_disk_written_bytes_total{device!~"dm-.*"}[5m]))',
        step='30s'
    )
    for series in results:
        instance = series['metric'].get('instance', '?').split(':')[0]
        timestamps = [datetime.fromtimestamp(float(v[0])) for v in series['values']]
        values = [float(v[1]) / 1e6 for v in series['values']]
        ax2.plot(timestamps, values, label=instance, linewidth=1.5)
    
    ax2.set_xlabel('Time')
    ax2.set_ylabel('Write (MB/s)')
    ax2.set_title('Disk Write I/O by Node')
    ax2.legend(fontsize=9)
    ax2.grid(alpha=0.3)
    
    fig.autofmt_xdate()
    plt.tight_layout()
    plt.show()
else:
    print('PROM_URL not set.')

---
## 5. Node Summary Dashboard

Let's build a text-based summary that combines key metrics from all nodes.

In [ ]:
if PROM_URL:
    print('=' * 80)
    print('NODE RESOURCE SUMMARY')
    print('=' * 80)
    
    # Gather all metrics in one pass
    queries = {
        'cpu_pct': '100 - (avg by(instance) (rate(node_cpu_seconds_total{mode="idle"}[5m])) * 100)',
        'mem_pct': '(1 - (node_memory_MemAvailable_bytes / node_memory_MemTotal_bytes)) * 100',
        'mem_total_gb': 'node_memory_MemTotal_bytes / (1024*1024*1024)',
        'mem_avail_gb': 'node_memory_MemAvailable_bytes / (1024*1024*1024)',
        'disk_pct': '(1 - (node_filesystem_avail_bytes{mountpoint="/",fstype!="rootfs"} / node_filesystem_size_bytes{mountpoint="/",fstype!="rootfs"})) * 100',
        'disk_avail_gb': 'node_filesystem_avail_bytes{mountpoint="/",fstype!="rootfs"} / (1024*1024*1024)',
        'load1': 'node_load1',
        'cores': 'count by(instance) (node_cpu_seconds_total{mode="idle"})',
        'uptime_hrs': '(time() - node_boot_time_seconds) / 3600',
        'net_rx_mbps': 'sum by(instance) (rate(node_network_receive_bytes_total{device!~"lo|veth.*|docker.*|br-.*"}[5m])) / 1e6',
        'net_tx_mbps': 'sum by(instance) (rate(node_network_transmit_bytes_total{device!~"lo|veth.*|docker.*|br-.*"}[5m])) / 1e6',
    }
    
    # Collect all data keyed by instance
    data = {}
    for metric_name, query in queries.items():
        for r in prom_query(query):
            instance = r['metric'].get('instance', '?')
            if instance not in data:
                data[instance] = {}
            data[instance][metric_name] = float(r['value'][1])
    
    # Display
    for instance in sorted(data.keys()):
        d = data[instance]
        short = instance.split(':')[0]
        cores = int(d.get('cores', 0))
        print(f'\n--- {short} ({cores} cores) ---')
        print(f'  CPU:     {d.get("cpu_pct", 0):>6.1f}%  (load1: {d.get("load1", 0):.2f})')
        print(f'  Memory:  {d.get("mem_pct", 0):>6.1f}%  ({d.get("mem_avail_gb", 0):.1f} / {d.get("mem_total_gb", 0):.1f} GB available)')
        print(f'  Disk:    {d.get("disk_pct", 0):>6.1f}%  ({d.get("disk_avail_gb", 0):.1f} GB free)')
        print(f'  Network: RX {d.get("net_rx_mbps", 0):.2f} MB/s | TX {d.get("net_tx_mbps", 0):.2f} MB/s')
        print(f'  Uptime:  {d.get("uptime_hrs", 0):.1f} hours')
    
    print(f'\n{"=" * 80}')
else:
    print('PROM_URL not set.')

---
## 6. CPU Mode Breakdown

CPU time is split across modes: user, system, idle, iowait, irq, softirq, etc. This breakdown shows where CPU time is being spent.

In [ ]:
if PROM_URL:
    modes_of_interest = ['user', 'system', 'iowait', 'idle']
    
    fig, axes = plt.subplots(1, len(NODE_IPS), figsize=(5 * len(NODE_IPS), 5))
    if len(NODE_IPS) == 1:
        axes = [axes]  # Ensure iterable
    
    for ax, (node_name, node_ip) in zip(axes, NODE_IPS.items()):
        instance = f'{node_ip}:9100'
        mode_values = []
        mode_labels = []
        
        for mode in modes_of_interest:
            results = prom_query(
                f'avg(rate(node_cpu_seconds_total{{instance="{instance}",mode="{mode}"}}[5m])) * 100'
            )
            if results:
                value = float(results[0]['value'][1])
                if value > 0.01:  # Skip negligible values
                    mode_values.append(value)
                    mode_labels.append(mode)
        
        if mode_values:
            colors = {'user': '#4C72B0', 'system': '#DD8452', 'iowait': '#C44E52', 'idle': '#55A868'}
            bar_colors = [colors.get(m, '#999999') for m in mode_labels]
            ax.bar(mode_labels, mode_values, color=bar_colors, alpha=0.8)
            ax.set_ylabel('CPU Time (%)')
            ax.set_title(f'{node_name}')
            ax.grid(axis='y', alpha=0.3)
        else:
            ax.text(0.5, 0.5, 'No data', ha='center', va='center', transform=ax.transAxes)
            ax.set_title(f'{node_name}')
    
    plt.suptitle('CPU Mode Breakdown by Node', fontsize=14, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print('PROM_URL not set.')

---
## 7. Resource Utilization Snapshot (Bar Chart)

A side-by-side comparison of CPU, memory, and disk usage across all nodes.

In [ ]:
if PROM_URL:
    metrics = {
        'CPU %': '100 - (avg by(instance) (rate(node_cpu_seconds_total{mode="idle"}[5m])) * 100)',
        'Memory %': '(1 - (node_memory_MemAvailable_bytes / node_memory_MemTotal_bytes)) * 100',
        'Disk %': '(1 - (node_filesystem_avail_bytes{mountpoint="/",fstype!="rootfs"} / node_filesystem_size_bytes{mountpoint="/",fstype!="rootfs"})) * 100',
    }
    
    # Collect data
    chart_data = {}  # {instance: {metric: value}}
    for metric_name, query in metrics.items():
        for r in prom_query(query):
            instance = r['metric'].get('instance', '?').split(':')[0]
            if instance not in chart_data:
                chart_data[instance] = {}
            chart_data[instance][metric_name] = float(r['value'][1])
    
    if chart_data:
        instances = sorted(chart_data.keys())
        metric_names = list(metrics.keys())
        
        x = np.arange(len(instances))
        width = 0.25
        
        fig, ax = plt.subplots(figsize=(10, 5))
        colors = ['#4C72B0', '#DD8452', '#55A868']
        
        for i, metric in enumerate(metric_names):
            values = [chart_data[inst].get(metric, 0) for inst in instances]
            ax.bar(x + i * width, values, width, label=metric, color=colors[i], alpha=0.8)
        
        ax.set_xlabel('Node')
        ax.set_ylabel('Utilization (%)')
        ax.set_title('Resource Utilization by Node')
        ax.set_xticks(x + width)
        ax.set_xticklabels(instances)
        ax.set_ylim(0, 100)
        ax.legend()
        ax.grid(axis='y', alpha=0.3)
        plt.tight_layout()
        plt.show()
    else:
        print('No data returned.')
else:
    print('PROM_URL not set.')

---
## 8. Exporting Data to CSV

You can export Prometheus range query results to CSV for use in other tools (Excel, R, etc.).

In [ ]:
import csv
from io import StringIO

if PROM_URL:
    # Export CPU utilization to CSV
    results = prom_query_range(
        '100 - (avg by(instance) (rate(node_cpu_seconds_total{mode="idle"}[5m])) * 100)',
        step='60s'  # 1-minute resolution for manageable file size
    )
    
    if results:
        output = StringIO()
        writer = csv.writer(output)
        writer.writerow(['timestamp', 'instance', 'cpu_pct'])
        
        for series in results:
            instance = series['metric'].get('instance', '?').split(':')[0]
            for ts, val in series['values']:
                dt = datetime.fromtimestamp(float(ts)).isoformat()
                writer.writerow([dt, instance, f'{float(val):.2f}'])
        
        csv_text = output.getvalue()
        
        # Save to file
        csv_path = os.path.join(STATUS_DIR, 'cpu_utilization.csv')
        with open(csv_path, 'w') as f:
            f.write(csv_text)
        print(f'Saved to {csv_path}')
        
        # Preview first 10 lines
        print('\nPreview:')
        for line in csv_text.strip().split('\n')[:10]:
            print(f'  {line}')
        print(f'  ... ({csv_text.count(chr(10))} rows total)')
    else:
        print('No data returned.')
else:
    print('PROM_URL not set.')

---
## 9. Understanding PromQL

PromQL (Prometheus Query Language) is how you extract data from Prometheus. Here are the key concepts:

### Selectors

Select metrics by name and filter by labels:
```promql
node_cpu_seconds_total                           # All CPU time-series
node_cpu_seconds_total{mode="idle"}              # Only idle CPU
node_cpu_seconds_total{instance=~"10\\.135.*"}   # Regex match on instance
node_cpu_seconds_total{mode!="idle"}             # Exclude idle
```

### Functions

| Function | Purpose | Example |
|----------|---------|---------|
| `rate(v[5m])` | Per-second rate of change over 5 minutes | `rate(node_cpu_seconds_total{mode="idle"}[5m])` |
| `avg by(label)` | Average across instances | `avg by(instance) (rate(...))` |
| `sum by(label)` | Sum across dimensions | `sum by(instance) (rate(node_network_receive_bytes_total[5m]))` |
| `topk(n, v)` | Top N series by value | `topk(1, node_load1)` |
| `time()` | Current Unix timestamp | `time() - node_boot_time_seconds` (uptime) |
| `count by(label)` | Count series per group | `count by(instance) (node_cpu_seconds_total{mode="idle"})` (core count) |

### Common Patterns

**CPU utilization** (counter → percentage):
```promql
100 - (avg by(instance) (rate(node_cpu_seconds_total{mode="idle"}[5m])) * 100)
```
Explanation: `rate()` converts the counter to per-second change. `avg by(instance)` averages across all CPU cores. Subtracting from 100 converts idle → busy.

**Memory utilization** (gauges → percentage):
```promql
(1 - (node_memory_MemAvailable_bytes / node_memory_MemTotal_bytes)) * 100
```
Explanation: Available/Total gives the free fraction. Subtract from 1 for used fraction.

**Network throughput** (counter → bytes/sec):
```promql
rate(node_network_receive_bytes_total{device!~"lo|veth.*"}[5m])
```
Explanation: `rate()` on a counter gives per-second rate. Filter out loopback and virtual interfaces.

---
## 10. Custom PromQL Playground

Use this cell to experiment with your own PromQL queries.

In [ ]:
# === Custom Query ===
# Edit the query below and run this cell

MY_QUERY = 'node_load1'

if PROM_URL:
    results = prom_query(MY_QUERY)
    if results:
        print(f'Query: {MY_QUERY}')
        print(f'Results: {len(results)} time-series\n')
        for r in results:
            labels = r['metric']
            value = r['value'][1]
            # Format labels as key=value pairs
            label_str = ', '.join(f'{k}={v}' for k, v in sorted(labels.items()) if k != '__name__')
            print(f'  {labels.get("__name__", "?")}{{{label_str}}} = {value}')
    else:
        print(f'No results for: {MY_QUERY}')
else:
    print('PROM_URL not set.')

In [ ]:
# === Custom Range Query with Chart ===
# Edit the query and time range, then run this cell

MY_RANGE_QUERY = 'node_load1'
LOOKBACK_MINUTES = 60  # How many minutes of history to show
STEP = '30s'           # Resolution

if PROM_URL:
    from datetime import timedelta
    now = datetime.now(timezone.utc)
    start_ts = (now - timedelta(minutes=LOOKBACK_MINUTES)).timestamp()
    end_ts = now.timestamp()
    
    results = prom_query_range(MY_RANGE_QUERY, start=start_ts, end=end_ts, step=STEP)
    
    if results:
        fig, ax = plt.subplots(figsize=(12, 5))
        for series in results:
            labels = series['metric']
            instance = labels.get('instance', '?').split(':')[0]
            extra = {k: v for k, v in labels.items() if k not in ('__name__', 'instance', 'job')}
            label = instance
            if extra:
                label += ' ' + ','.join(f'{k}={v}' for k, v in extra.items())
            
            timestamps = [datetime.fromtimestamp(float(v[0])) for v in series['values']]
            values = [float(v[1]) for v in series['values']]
            ax.plot(timestamps, values, label=label, linewidth=1.5)
        
        ax.set_xlabel('Time')
        ax.set_ylabel('Value')
        ax.set_title(f'{MY_RANGE_QUERY} (last {LOOKBACK_MINUTES} min)')
        ax.legend(fontsize=8)
        ax.grid(alpha=0.3)
        fig.autofmt_xdate()
        plt.tight_layout()
        plt.show()
    else:
        print(f'No results for: {MY_RANGE_QUERY}')
else:
    print('PROM_URL not set.')

---
## 11. Adapting This for Your Own Experiments

### Adding Monitoring to Your Slice

You can reuse the monitoring components from this weave in your own FABRIC experiments:

**1. Install node_exporter on your VMs**

Copy and run the `tools/setup-exporter.sh` script on any FABRIC VM. It installs node_exporter as a systemd service listening on port 9100.

**2. Point Prometheus at your nodes**

If you have a running Prometheus instance (from this weave or your own), add your nodes to `/etc/prometheus/prometheus.yml`:

```yaml
scrape_configs:
  - job_name: my-experiment
    static_configs:
      - targets: ['10.x.x.x:9100', '10.y.y.y:9100']
```

Then restart Prometheus: `sudo systemctl restart prometheus`

**3. Add application metrics**

If your application exposes Prometheus metrics (e.g., a Python Flask app with `prometheus_client`), add it as a separate scrape job:

```yaml
  - job_name: my-app
    static_configs:
      - targets: ['10.x.x.x:8080']
```

### Querying from Your Own Notebooks

You can query the Prometheus API from any notebook or script that has network access to the monitor node. Just use the `prom_query()` and `prom_query_range()` functions defined above — copy them into your own notebook and set `PROM_URL` to your Prometheus instance.

### Key Takeaways

- Prometheus scrapes targets every 15 seconds — no agent push needed
- node_exporter provides ~1000 metrics per node out of the box
- PromQL's `rate()` function is essential for counter metrics (CPU, network, disk I/O)
- Grafana dashboards are great for real-time monitoring; matplotlib is better for post-hoc analysis
- All data is accessible via the HTTP API — no special client library needed

---
## Additional Resources

- **README.md**: Full documentation for this weave, including troubleshooting and extension guide
- **Grafana Dashboard**: Access via the LoomAI Apps tab (or directly at `http://<monitor-ip>:3000`)
- **Prometheus docs**: Search for "prometheus.io/docs" for the official PromQL reference
- **node_exporter metrics**: See the full list by visiting `http://<any-node-ip>:9100/metrics`